# NBA Offensive Gravity Proxy — V6

**Goal:** Build a transparent, defensible proxy for NBA offensive gravity from public box-score and advanced statistics.

This version intentionally **does not train a supervised model against VORP**. VORP is not ground-truth gravity. Instead, the final score combines four interpretable components:

1. **Perimeter Gravity** — outside shooting volume + shooting credibility  
2. **Interior Pressure** — two-point and free-throw pressure  
3. **Playmaking Gravity** — passing creation + offensive responsibility  
4. **Offensive Impact** — offensive impact + scoring efficiency

The score is then stress-tested with sensitivity analysis and compared with PCA plus external advanced metrics for convergent validity.

> Important limitation: public box-score data cannot directly observe defender distance, double teams, off-ball movement, screen gravity, or defensive rotations. This is therefore a **gravity proxy**, not a definitive tracking-based gravity metric.


## 1. Upload source data

Upload the same two source CSVs used in V5:

- `NBA Player Advanced Stats_2024-25.csv`
- `NBA Player Stats_2024-25_Total.csv`

If your filenames are different, update them in the next cell.


In [ ]:
from google.colab import files
uploaded = files.upload()


Saving NBA Player Advanced Stats_2024-25.csv to NBA Player Advanced Stats_2024-25.csv


In [ ]:
from google.colab import files
uploaded = files.upload()

Saving NBA Player Stats_2024-25_Total.csv to NBA Player Stats_2024-25_Total.csv


In [ ]:
import pandas as pd
import numpy as np

ADV_FILE = "NBA Player Advanced Stats_2024-25.csv"
BASIC_FILE = "NBA Player Stats_2024-25_Total.csv"

adv = pd.read_csv(ADV_FILE)
basic = pd.read_csv(BASIC_FILE)

print("Advanced columns:")
print(list(adv.columns))
print("\nBasic columns:")
print(list(basic.columns))


Advanced columns:
['Player', 'PER', 'TS%', 'USG%', 'OWS', 'DWS', 'WS', 'WS/48', 'OBPM', 'DBPM', 'BPM', 'VORP']

Basic columns:
['Rk', 'Player', 'Age', 'Team', 'Pos', 'G', 'GS', 'MP', 'FG', 'FGA', 'FG%', '3P', '3PA', '3P%', '2P', '2PA', '2P%', 'eFG%', 'FT', 'FTA', 'FT%', 'ORB', 'DRB', 'TRB', 'AST', 'STL', 'BLK', 'TOV', 'PF', 'PTS', 'Trp-Dbl']


## 2. Merge and clean

We aggregate duplicate player rows after merging, then apply a minimum-minutes qualification filter.

**Why use a minutes threshold?**  
Minutes should not *increase* a player's gravity score, but tiny samples can create extreme rates. The threshold is only used to ensure a reasonable sample.


In [ ]:
df = pd.merge(adv, basic, on="Player", how="inner")
df = df.groupby("Player").mean(numeric_only=True).reset_index()

# Keep players with a meaningful season sample.
# 500 minutes is a transparent portfolio-project threshold; you can sensitivity-test it later.
MIN_MINUTES = 1000
df = df[df["MP"] >= MIN_MINUTES].copy()

print(f"Qualified players: {len(df)}")
df.head()


Qualified players: 256


,Player,PER,TS%,USG%,OWS,DWS,WS,WS/48,OBPM,DBPM,...,ORB,DRB,TRB,AST,STL,BLK,TOV,PF,PTS,Trp-Dbl
0,A.J. Green,9.2,0.621,12.6,1.6,1.3,2.8,0.082,-1.7,-0.4,...,18.0,156.0,174.0,108.0,37.0,7.0,40.0,157.0,541.0,0.0
3,Aaron Gordon,17.0,0.650,19.0,3.6,0.7,4.3,0.143,2.7,-1.5,...,80.0,167.0,247.0,164.0,23.0,14.0,73.0,82.0,748.0,0.0
5,Aaron Nesmith,14.6,0.653,17.4,1.9,1.0,2.9,0.126,0.0,-0.5,...,37.0,141.0,178.0,54.0,35.0,17.0,37.0,114.0,541.0,0.0
6,Aaron Wiggins,16.1,0.596,20.3,2.7,2.7,5.4,0.148,1.3,-0.2,...,81.0,214.0,295.0,134.0,60.0,18.0,69.0,101.0,914.0,0.0
11,Al Horford,12.9,0.563,13.8,2.0,2.4,4.4,0.129,0.1,1.1,...,79.0,290.0,369.0,128.0,36.0,51.0,46.0,81.0,538.0,0.0


## 3. Create rate statistics

Season totals such as 3PA can be distorted by games/minutes played.  
We therefore convert major volume statistics to **per-36-minute rates**.


In [ ]:
def per36(series, minutes):
    return (series / minutes.replace(0, np.nan)) * 36

df["3PA_per36"] = per36(df["3PA"], df["MP"])
df["2PA_per36"] = per36(df["2PA"], df["MP"])
df["FTA_per36"] = per36(df["FTA"], df["MP"])
df["AST_per36"] = per36(df["AST"], df["MP"])

# Use 3P% if present. Otherwise calculate it from made threes / attempts.
if "3P%" in df.columns:
    df["ThreeP_Pct"] = df["3P%"]
elif "3P" in df.columns:
    df["ThreeP_Pct"] = df["3P"] / df["3PA"].replace(0, np.nan)
else:
    raise ValueError("Need either a '3P%' column or a '3P' column to calculate shooting credibility.")

# True-shooting efficiency relative to this qualified-player sample.
df["TS_Above_Avg"] = df["TS%"] - df["TS%"].mean()

df[["Player", "3PA_per36", "ThreeP_Pct", "2PA_per36", "FTA_per36", "AST_per36", "TS_Above_Avg"]].head()


,Player,3PA_per36,ThreeP_Pct,2PA_per36,FTA_per36,AST_per36,TS_Above_Avg
0,A.J. Green,7.877034,0.427,1.323689,0.585895,2.343580,0.042414
3,Aaron Gordon,4.279198,0.436,8.085695,4.453352,4.080166,0.071414
5,Aaron Nesmith,6.251113,0.431,5.898486,2.564559,1.731077,0.074414
6,Aaron Wiggins,6.997706,0.383,8.029817,1.837156,2.766055,0.017414
11,Al Horford,6.813743,0.363,3.189873,0.824593,2.777577,-0.015586


## 4. Standardize inputs

The inputs use different units, so they cannot be meaningfully added in raw form.

A **z-score** expresses each statistic in standard-deviation units:

`z = (player value - league/sample mean) / sample standard deviation`


In [ ]:
from sklearn.preprocessing import StandardScaler

raw_features = [
    "3PA_per36",
    "ThreeP_Pct",
    "2PA_per36",
    "FTA_per36",
    "AST_per36",
    "USG%",
    "OBPM",
    "TS_Above_Avg",
]

model_df = df.replace([np.inf, -np.inf], np.nan).dropna(subset=raw_features).copy()

scaler = StandardScaler()
z = scaler.fit_transform(model_df[raw_features])

for i, feature in enumerate(raw_features):
    model_df[f"z_{feature}"] = z[:, i]

print(f"Players remaining after feature cleaning: {len(model_df)}")


Players remaining after feature cleaning: 252


## 5. Build the four gravity components

### Perimeter Gravity
- 70% three-point attempt rate per 36
- 30% three-point percentage

Volume matters more than percentage because defensive spacing pressure depends heavily on willingness to shoot, while percentage provides shooting credibility.

### Interior Pressure
- 60% two-point attempts per 36
- 40% free-throw attempts per 36

This is a public-data proxy for paint/rim pressure.

### Playmaking Gravity
- 70% assists per 36
- 30% usage percentage

Assists represent creation; usage adds context for how much offensive responsibility the player carries.

### Offensive Impact
- 70% OBPM
- 30% true-shooting efficiency relative to the sample

This keeps efficiency and offensive impact from overwhelming the more directly gravity-related components.


In [ ]:
model_df["Perimeter_Gravity"] = (
    0.70 * model_df["z_3PA_per36"] +
    0.30 * model_df["z_ThreeP_Pct"]
)

model_df["Interior_Pressure"] = (
    0.60 * model_df["z_2PA_per36"] +
    0.40 * model_df["z_FTA_per36"]
)

model_df["Playmaking_Gravity"] = (
    0.70 * model_df["z_AST_per36"] +
    0.30 * model_df["z_USG%"]
)

model_df["Offensive_Impact"] = (
    0.70 * model_df["z_OBPM"] +
    0.30 * model_df["z_TS_Above_Avg"]
)

component_cols = [
    "Perimeter_Gravity",
    "Interior_Pressure",
    "Playmaking_Gravity",
    "Offensive_Impact",
]

model_df[["Player"] + component_cols].head()


,Player,Perimeter_Gravity,Interior_Pressure,Playmaking_Gravity,Offensive_Impact
0,A.J. Green,0.853523,-1.722537,-0.934316,-0.248845
3,Aaron Gordon,-0.077125,0.336778,-0.001118,1.247097
5,Aaron Nesmith,0.432718,-0.469189,-0.888177,0.470192
6,Aaron Wiggins,0.468216,-0.246926,-0.380625,0.467416
11,Al Horford,0.350724,-1.335916,-0.722338,-0.110763


## 6. Final transparent Gravity Index

Baseline weights:

- **30% Perimeter Gravity**
- **25% Interior Pressure**
- **25% Playmaking Gravity**
- **20% Offensive Impact**

These are analyst-defined weights, not learned ground-truth coefficients.  
That is why we explicitly test their stability later rather than pretending the weights are objectively correct.


In [ ]:
BASE_WEIGHTS = {
    "Perimeter_Gravity": 0.30,
    "Interior_Pressure": 0.25,
    "Playmaking_Gravity": 0.25,
    "Offensive_Impact": 0.20,
}

model_df["Gravity_Index_Raw"] = sum(
    BASE_WEIGHTS[col] * model_df[col] for col in component_cols
)


## 7. Convert the raw index to a 0–100 presentation score

The 0–100 score is a **presentation layer**. It does not mean the top player has literally "100 units" of gravity.


In [ ]:
from sklearn.preprocessing import MinMaxScaler

score_scaler = MinMaxScaler(feature_range=(0, 100))
model_df["Gravity_Score_100"] = score_scaler.fit_transform(
    model_df[["Gravity_Index_Raw"]]
).ravel()

model_df = model_df.sort_values("Gravity_Score_100", ascending=False).reset_index(drop=True)

model_df[["Player", "Gravity_Score_100"] + component_cols].head(20)


,Player,Gravity_Score_100,Perimeter_Gravity,Interior_Pressure,Playmaking_Gravity,Offensive_Impact
0,Shai Gilgeous-Alexander,100.000000,0.181685,2.975332,1.745227,2.989651
1,Nikola Jokic,98.871897,-0.047028,1.887603,2.587026,3.461293
2,Luka Doncic,92.835651,1.159315,1.657616,2.055646,1.646543
3,LaMelo Ball,92.473134,1.822891,1.148299,2.330160,0.887211
4,Trae Young,88.813805,0.706197,1.256847,3.118302,0.861264
5,Stephen Curry,88.622549,1.995131,0.317281,1.477845,2.122572
6,Giannis Antetokounmpo,85.651650,-1.687011,3.888520,1.792291,2.317694
7,Cade Cunningham,84.511248,0.145979,1.835549,2.558925,0.995338
8,Ja Morant,83.103257,0.150375,2.133834,2.253004,0.775076
9,James Harden,81.373847,0.796454,0.997604,2.197011,1.022068


## 8. Assign percentile-based Gravity Tiers

Percentile tiers are more robust than fixed 85/65/45 cutoffs because min-max scores depend on the strongest and weakest players in a particular season.

- Top 5%: Elite Gravity
- 5–15%: High Gravity
- 15–50%: Starter Gravity
- Bottom 50%: Rotational Gravity


In [ ]:
pct = model_df["Gravity_Score_100"].rank(pct=True, ascending=True)

def gravity_tier(percentile):
    if percentile >= 0.95:
        return "Elite Gravity"
    elif percentile >= 0.85:
        return "High Gravity"
    elif percentile >= 0.50:
        return "Starter Gravity"
    else:
        return "Rotational Gravity"

model_df["Gravity_Tier"] = pct.apply(gravity_tier)

model_df[["Player", "Gravity_Score_100", "Gravity_Tier"]].head(25)


,Player,Gravity_Score_100,Gravity_Tier
0,Shai Gilgeous-Alexander,100.000000,Elite Gravity
1,Nikola Jokic,98.871897,Elite Gravity
2,Luka Doncic,92.835651,Elite Gravity
3,LaMelo Ball,92.473134,Elite Gravity
4,Trae Young,88.813805,Elite Gravity
5,Stephen Curry,88.622549,Elite Gravity
6,Giannis Antetokounmpo,85.651650,Elite Gravity
7,Cade Cunningham,84.511248,Elite Gravity
8,Ja Morant,83.103257,Elite Gravity
9,James Harden,81.373847,Elite Gravity


## 9. Sensitivity analysis

A custom index should not collapse if the weights move slightly.

We test several plausible alternative weighting schemes and compare each ranking with the baseline using **Spearman rank correlation** and Top-20 overlap.


In [ ]:
from scipy.stats import spearmanr

weight_scenarios = {
    "Baseline": [0.30, 0.25, 0.25, 0.20],
    "More_Perimeter": [0.40, 0.20, 0.20, 0.20],
    "More_Interior": [0.25, 0.35, 0.20, 0.20],
    "More_Playmaking": [0.25, 0.20, 0.35, 0.20],
    "More_Impact": [0.25, 0.20, 0.20, 0.35],
    "Equal": [0.25, 0.25, 0.25, 0.25],
}

baseline = model_df.set_index("Player")["Gravity_Index_Raw"]
baseline_top20 = set(baseline.nlargest(20).index)

sensitivity_rows = []

for name, weights in weight_scenarios.items():
    alt = sum(w * model_df[col] for w, col in zip(weights, component_cols))
    alt_series = pd.Series(alt.values, index=model_df["Player"])

    rho, _ = spearmanr(baseline.loc[alt_series.index], alt_series)
    top20_overlap = len(baseline_top20 & set(alt_series.nlargest(20).index))

    sensitivity_rows.append({
        "Scenario": name,
        "Spearman_vs_Baseline": rho,
        "Top20_Overlap": top20_overlap
    })

sensitivity_df = pd.DataFrame(sensitivity_rows)
sensitivity_df


,Scenario,Spearman_vs_Baseline,Top20_Overlap
0,Baseline,1.000000,20
1,More_Perimeter,0.976564,18
2,More_Interior,0.983310,19
3,More_Playmaking,0.992854,19
4,More_Impact,0.976501,19
5,Equal,0.994123,19


## 10. PCA robustness comparison

PCA is **not** used to define the final gravity score.  
It is used as an unsupervised comparison to see whether a data-derived latent factor broadly agrees with the transparent composite.


In [ ]:
from sklearn.decomposition import PCA

component_scaler = StandardScaler()
X_components = component_scaler.fit_transform(model_df[component_cols])

pca = PCA(n_components=1)
pc1 = pca.fit_transform(X_components).ravel()

# Orient PC1 so that higher values correspond to higher baseline gravity.
if np.corrcoef(pc1, model_df["Gravity_Index_Raw"])[0, 1] < 0:
    pc1 = -pc1

model_df["PCA_Gravity_Factor"] = pc1

pca_rho, _ = spearmanr(
    model_df["Gravity_Index_Raw"],
    model_df["PCA_Gravity_Factor"]
)

print("PC1 explained variance:", round(pca.explained_variance_ratio_[0], 3))
print("Spearman correlation, PCA factor vs baseline:", round(pca_rho, 3))


PC1 explained variance: 0.517
Spearman correlation, PCA factor vs baseline: 0.877


## 11. Convergent validation against advanced metrics

VORP/BPM are **not ground-truth gravity** and are not used as the target.

They can still be useful as external comparison metrics: if the gravity proxy has no relationship at all with established offensive/value measures, that would be a reason to inspect the index.


In [ ]:
validation_metrics = [c for c in ["VORP", "BPM"] if c in model_df.columns]

validation_rows = []
for metric in validation_metrics:
    rho, _ = spearmanr(model_df["Gravity_Score_100"], model_df[metric], nan_policy="omit")
    validation_rows.append({
        "Metric": metric,
        "Spearman_with_Gravity": rho
    })

validation_df = pd.DataFrame(validation_rows)
validation_df


,Metric,Spearman_with_Gravity
0,VORP,0.627124
1,BPM,0.599670


## 12. Inspect the leaders and component profiles

This is the basketball sanity check: do the top players make conceptual sense, and *why* do they score highly?


In [ ]:
leader_cols = [
    "Player",
    "Gravity_Score_100",
    "Gravity_Tier",
    "Perimeter_Gravity",
    "Interior_Pressure",
    "Playmaking_Gravity",
    "Offensive_Impact",
    "3PA_per36",
    "ThreeP_Pct",
    "TS%",
    "USG%",
    "OBPM",
]

display(model_df[leader_cols].head(25))


,Player,Gravity_Score_100,Gravity_Tier,Perimeter_Gravity,Interior_Pressure,Playmaking_Gravity,Offensive_Impact,3PA_per36,ThreeP_Pct,TS%,USG%,OBPM
0,Shai Gilgeous-Alexander,100.000000,Elite Gravity,0.181685,2.975332,1.745227,2.989651,6.027714,0.375,0.637,34.800000,8.9
1,Nikola Jokic,98.871897,Elite Gravity,-0.047028,1.887603,2.587026,3.461293,4.634772,0.417,0.663,29.500000,9.9
2,Luka Doncic,92.835651,Elite Gravity,1.159315,1.657616,2.055646,1.646543,9.788581,0.367,0.587,33.833333,5.5
3,LaMelo Ball,92.473134,Elite Gravity,1.822891,1.148299,2.330160,0.887211,12.629900,0.339,0.536,35.900000,4.1
4,Trae Young,88.813805,Elite Gravity,0.706197,1.256847,3.118302,0.861264,8.438116,0.340,0.567,29.600000,3.3
5,Stephen Curry,88.622549,Elite Gravity,1.995131,0.317281,1.477845,2.122572,12.532860,0.397,0.618,29.800000,6.4
6,Giannis Antetokounmpo,85.651650,Elite Gravity,-1.687011,3.888520,1.792291,2.317694,0.990826,0.222,0.625,35.200000,6.9
7,Cade Cunningham,84.511248,Elite Gravity,0.145979,1.835549,2.558925,0.995338,6.137031,0.356,0.565,33.200000,3.8
8,Ja Morant,83.103257,Elite Gravity,0.150375,2.133834,2.253004,0.775076,6.754444,0.309,0.563,32.200000,3.1
9,James Harden,81.373847,Elite Gravity,0.796454,0.997604,2.197011,1.022068,8.622445,0.352,0.582,29.600000,3.5


## 13. Export Tableau-ready dataset

This export contains the final score **and its component scores**, so Tableau can explain *why* each player has the gravity score they do.


In [ ]:
from google.colab import files

tableau_cols = [
    "Player",
    "Gravity_Score_100",
    "Gravity_Tier",
    "Perimeter_Gravity",
    "Interior_Pressure",
    "Playmaking_Gravity",
    "Offensive_Impact",
    "3PA_per36",
    "ThreeP_Pct",
    "2PA_per36",
    "FTA_per36",
    "AST_per36",
    "TS%",
    "USG%",
    "OBPM",
    "PCA_Gravity_Factor",
]

for optional in ["VORP", "BPM"]:
    if optional in model_df.columns:
        tableau_cols.append(optional)

dashboard_df = model_df[tableau_cols].copy()

OUTPUT_FILE = "nba_gravity_v6_tableauFINAL.csv"
dashboard_df.to_csv(OUTPUT_FILE, index=False)

print(f"Saved {len(dashboard_df)} players to {OUTPUT_FILE}")
display(dashboard_df.head(20))

files.download(OUTPUT_FILE)


Saved 252 players to nba_gravity_v6_tableauFINAL.csv


,Player,Gravity_Score_100,Gravity_Tier,Perimeter_Gravity,Interior_Pressure,Playmaking_Gravity,Offensive_Impact,3PA_per36,ThreeP_Pct,2PA_per36,FTA_per36,AST_per36,TS%,USG%,OBPM,PCA_Gravity_Factor,VORP,BPM
0,Shai Gilgeous-Alexander,100.000000,Elite Gravity,0.181685,2.975332,1.745227,2.989651,6.027714,0.375,16.919169,9.270208,6.734411,0.637,34.800000,8.9,4.831707,8.9,11.5
1,Nikola Jokic,98.871897,Elite Gravity,-0.047028,1.887603,2.587026,3.461293,4.634772,0.417,14.464411,6.315053,10.025671,0.663,29.500000,9.9,4.971453,9.8,13.3
2,Luka Doncic,92.835651,Elite Gravity,1.159315,1.657616,2.055646,1.646543,9.788581,0.367,11.070661,8.038440,7.794234,0.587,33.833333,5.5,3.445997,2.6,6.7
3,LaMelo Ball,92.473134,Elite Gravity,1.822891,1.148299,2.330160,0.887211,12.629900,0.339,11.362126,5.477741,8.276412,0.536,35.900000,4.1,2.881506,2.0,3.2
4,Trae Young,88.813805,Elite Gravity,0.706197,1.256847,3.118302,0.861264,8.438116,0.340,9.647317,7.373494,11.566265,0.567,29.600000,3.3,3.389187,1.7,0.5
5,Stephen Curry,88.622549,Elite Gravity,1.995131,0.317281,1.477845,2.122572,12.532860,0.397,7.577265,4.779751,6.730018,0.618,29.800000,6.4,2.558628,4.8,6.3
6,Giannis Antetokounmpo,85.651650,Elite Gravity,-1.687011,3.888520,1.792291,2.317694,0.990826,0.222,19.753604,11.119266,6.809961,0.625,35.200000,6.9,4.928987,6.6,9.5
7,Cade Cunningham,84.511248,Elite Gravity,0.145979,1.835549,2.558925,0.995338,6.137031,0.356,15.254486,5.432300,9.367047,0.565,33.200000,3.8,3.438110,3.7,3.9
8,Ja Morant,83.103257,Elite Gravity,0.150375,2.133834,2.253004,0.775076,6.754444,0.309,14.314681,7.560237,8.626728,0.563,32.200000,3.1,3.292498,1.7,2.4
9,James Harden,81.373847,Elite Gravity,0.796454,0.997604,2.197011,1.022068,8.622445,0.352,8.093223,7.460739,8.867695,0.582,29.600000,3.5,2.725850,4.4,4.3


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# Interpretation / README Notes

When presenting this project:

**What it is:**  
A transparent public-data proxy for offensive gravity.

**What it is not:**  
A direct measurement of defender attention or tracking-based gravity.

**Why no supervised target?**  
There is no reliable public ground-truth gravity label in the source data. Training against VORP would teach the model to predict VORP rather than gravity.

**Key methodological strengths:**  
- Per-36 volume rates reduce playing-time distortion  
- Four interpretable basketball components  
- Standardization puts inputs on comparable scales  
- Sensitivity analysis tests weight robustness  
- PCA provides an unsupervised robustness comparison  
- VORP/BPM are external comparisons rather than targets  
- Component-level exports make the Tableau dashboard explainable
